# 14 · Solar/diurnal context, dependence-aware nulls and a sensitive adequacy auditor

**Why** (review 5). The projection-free protocol of notebook 11 failed its validation. The main failure (Type I 0.787) was a diurnal cycle that changes over the season, encoded in the image; no clock-time context level could represent it. Intervals also under-covered when information persisted for weeks, and the adequacy diagnostic never fired. Notebooks 10–13 stay unchanged as the record.

**What is new here.**
- **Context candidates**, chosen per split by week-grouped inner CV on training rows only (`inference.select_capacity`). The family is kept modest:
  - L3: the notebook 11 clock-time context;
  - L4: L3 + regularised hour × season products;
  - S: L3 + physically informed solar terms (`data.solar`): solar-elevation spline, elevation × T, signed hour angle × declination, and a Fourier basis of the daylight fraction (sunrise-to-sunset time);
  - S4: S + the L4 products.

  Local clock time stays in every candidate. Solar elevation is not assumed to carry the whole diurnal state.
- **Two null generators for Q0**, both refitting the joint model:
  - *shift*: the cyclic shift within seasons (39 evenly spaced shifts of ≥ 7 days);
  - *CRT*: a conditional randomisation test. V* = cross-fitted E^[V | raw context, season] plus a simulated residual process, with a day-level AR(1) across days and an AR(1) within days fitted per season, and innovations correlated across features (`inference.conditional_sampler`). Labels are never read.
  - Both generators keep season-specific visual structure: the shift generator projects on the richest basis plus season × day-of-year curves, and the CRT model includes the season.
- **Adequacy auditors**, run on the chosen context and on a deliberately misspecified one (L1). Each is scored on the same held-out folds and flags when its week-cluster one-sided 95 % lower bound exceeds a materiality threshold τ, chosen in §2.
  - *Primary:* a context-only challenger (`inference.challenger`): the audited design plus smooth random Fourier features of the raw context, with length scales fixed in advance (2 h, 3 K, 20 days, 10°, 0.15 of the daylight period), in one ridge with a grouped-CV penalty.
  - *Secondary:* the repair gain (`inference.repair_gain`) of adding a train-only *smooth* projection E[V | C] (a ridge on the richest basis plus the same features) to the context. It is a diagnostic only.
  - Both replaced first versions that failed in development (§2).
- **Intervals.** Week-cluster intervals are no longer used for confirmation. The candidate interval is a t interval over the per-season held-out means of leave-one-season-out (season clusters, 3 df).
  - Its target is **G for this design**: the expected future-season gain of models fitted on three seasons, estimated per world by the frozen chronological models scored on three further seasons. Coverage of E2 is reported beside it, because G ≠ E2 (statistical note §2).
  - Season-specific G (fit on seasons 1–3, evaluate season 4) is reported descriptively, without an interval.

**Design.** 4 seasons × 92 days × 14 hourly rows at CPER's location (40.8° N, UTC−7), as planned for CPER 2020–2023. There are no NEON observations in this notebook.

**Protocol.**
- §1 development worlds (seeds 600 000+, a separate range per world) and §2 the development reading.
- The gates, the chosen null generator and the auditor thresholds are declared in notebook 16 before its validation run, on new worlds (seeds 700 000+).
- The simulation machinery and the protocol live in `visual_hygrometry.suite`, so that notebook 16 runs exactly the code developed here.
- The notebook 11 worlds are not reused, except that the catastrophic season × hour null is included again on new seeds, as review 5 asked.

In [1]:
import os
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')

import numpy as np
import pandas as pd
from joblib import Parallel, delayed

from visual_hygrometry import inference, scoring, splits, suite, worlds
from visual_hygrometry.suite import N_NULL, N_SEAS, DOY0, KNOTS, DOY_K, DOY_F, cmi, encodes, gam, make, pipeline, regimes, season_x_hour, spec, sun, sun_clock, vegetation

pd.set_option('display.precision', 3)
par = lambda: Parallel(n_jobs=16, max_nbytes=None)

## 0 · Truth of the shared-regime positive world

Weather regimes (a two-state Markov chain over days) shift both D and the image, and they are not part of C. Given C, the pair (D residual, V) is a two-component mixture, so I(D; V | C) is computed by Monte Carlo from the exact mixture densities.

In [2]:
print('shared-regime truth (nats):', round(suite.truth_shared_regime(.5, (0, -2), (2, 3), (0, 1)), 4))

shared-regime truth (nats): 0.0187


## 1 · Development worlds

Every world has the solar climate: T and D follow lagged insolation, D has a dawn minimum tied to sunrise, and sunrise and day length change across June–August. 40 replicates per world, 4 seasons, each world with its own seeds.

| world | structure | truth |
|---|---|---|
| D-N1 | solar climate only; image unrelated | 0 |
| D-N2 | image = brightness ∝ sin(solar elevation) + noise (a "sun clock") | 0 |
| D-N3 | the notebook 11 catastrophic null: D += 0.8 sin(hour) tanh(season); image encodes it | 0 |
| D-N4 | D += 0.12 (T − 20) × insolation; image encodes it | 0 |
| D-N5 | moisture residual and image noise persistence 0.95 | 0 |
| D-N6 | persistence 0.98 | 0 |
| D-N7 | independent Markov weather regimes in D (mean −2 K, SD × 1.5) and in the image (mean +1), each persisting 0.9–0.95 per day | 0 |
| D-N8 | season-specific greening (random amplitude and timing per season) and a camera change from season 3 | 0 |
| D-P1 | image sees the moisture residual | 0.03 |
| D-P2 | slow shared state | exact |
| D-P3 | sun-clock image that also sees the moisture residual | 0.03 |
| D-P4 | shared Markov weather regimes shift D and the image | Monte Carlo |

In [3]:
txs = lambda d: 0.12 * (d.temp_cond_c - 20) * sun(d, 0.0)
# shared regime: residual mixture of N(0, 2) and N(-2, 3); image means 0 and 1; Markov stationary probability 0.5
DEV = {'D-N1': dict(spec=spec()), 'D-N2': dict(spec=spec(), modify=sun_clock(1.5, .3)), 'D-N3': dict(spec=spec(), modify=encodes(season_x_hour, 1.5, .3), vis=['vis_ctx']),
       'D-N4': dict(spec=spec(), modify=encodes(txs, 1.0, .3), vis=['vis_ctx']),
       'D-N5': dict(spec=spec(r_persist=.95, v_persist=.95, r_day=.8, v_day=.8)), 'D-N6': dict(spec=spec(r_persist=.98, v_persist=.98, r_day=.8, v_day=.8)),
       'D-N7': dict(spec=spec(), modify=regimes(.9, .95)), 'D-N8': dict(spec=spec(), modify=vegetation()),
       'D-P1': dict(spec=spec(gamma=gam(.03)), truth=cmi(spec(gamma=gam(.03)))), 'D-P2': dict(spec=spec(slow=(1.2, .7)), truth=cmi(spec(slow=(1.2, .7)))),
       'D-P3': dict(spec=spec(gamma=gam(.03)), modify=sun_clock(1.5, .3), truth=cmi(spec(gamma=gam(.03)))),  # scaling V leaves I unchanged
       'D-P4': dict(spec=spec(), modify=regimes(.9, .9, shared=True), truth=suite.truth_shared_regime(.5, (0, -2), (2, 3), (0, 1)))}
DEV = {k: {'vis': ['vis_moist'], 'truth': 0.0} | W for k, W in DEV.items()}
print({k: round(W['truth'], 4) for k, W in DEV.items()})
dev_raw = suite.run(DEV, 600000, 40)
dev = suite.summarise(DEV, dev_raw)
dev

{'D-N1': 0.0, 'D-N2': 0.0, 'D-N3': 0.0, 'D-N4': 0.0, 'D-N5': 0.0, 'D-N6': 0.0, 'D-N7': 0.0, 'D-N8': 0.0, 'D-P1': np.float64(0.03), 'D-P2': np.float64(0.0455), 'D-P3': np.float64(0.03), 'D-P4': np.float64(0.0187)}


,truth,G_design,rej_shift,rej_crt,est,bias,est_minus_G,cover_G,cover_E2,cover_week_G,chrono_G,dest_rand_q90,dest_rare_q90,audit,audit_L1,repair,repair_L1,level,failed
world,,,,,,,,,,,,,,,,,,,
D-N1,0.000,-1.695e-04,0.050,0.025,-1.379e-04,-1.379e-04,3.163e-05,0.975,1.000,0.950,1.702e-04,2.967e-03,2.106e-03,0.050,0.000,0.075,0.050,2.587,0
D-N2,0.000,-2.456e-04,0.050,0.025,-3.924e-04,-3.924e-04,-1.468e-04,0.975,0.975,0.925,-9.216e-04,1.826e-03,1.500e-03,0.000,0.025,0.100,0.100,2.694,0
D-N3,0.000,1.077e-04,0.025,0.025,2.515e-04,2.515e-04,1.437e-04,1.000,0.950,0.975,1.732e-04,1.362e-03,6.680e-04,0.075,0.550,0.275,1.000,2.850,0
D-N4,0.000,-8.908e-05,0.075,0.050,-1.981e-04,-1.981e-04,-1.090e-04,0.975,0.950,0.950,5.633e-05,9.056e-04,3.442e-04,0.050,0.000,0.025,0.025,2.712,0
D-N5,0.000,-1.957e-02,0.175,0.025,-1.352e-02,-1.352e-02,6.045e-03,0.775,0.925,0.525,-2.304e-02,1.814e-02,1.566e-02,0.100,0.150,0.125,0.100,2.700,0
D-N6,0.000,-2.501e-02,0.100,0.025,-3.659e-02,-3.659e-02,-1.158e-02,0.825,0.975,0.400,-2.364e-02,2.362e-02,1.976e-02,0.050,0.075,0.075,0.125,2.788,0
D-N7,0.000,4.812e-04,0.050,0.050,-3.483e-04,-3.483e-04,-8.295e-04,0.900,0.975,0.675,-4.098e-04,1.659e-03,2.113e-03,0.025,0.025,0.050,0.025,2.538,0
D-N8,0.000,7.488e-05,0.125,0.075,3.889e-04,3.889e-04,3.140e-04,0.925,0.925,0.875,6.867e-04,2.770e-03,2.812e-03,0.050,0.025,0.150,0.050,2.575,0
D-P1,0.030,2.245e-02,1.000,1.000,2.319e-02,-6.809e-03,7.394e-04,0.900,0.750,0.775,2.550e-02,6.104e-03,8.053e-03,0.000,0.000,0.100,0.000,2.700,0


### Clock-time vs solar context, forced (development diagnostic)

The same three diurnal nulls and the sun-clock positive, with the context forced to one candidate (no capacity search). 40 replicates; shift test only.

In [4]:
def forced_rep(W, level, seed):
    df = make(W, N_SEAS, seed)
    out = pipeline(df, W, full=False, force=level)
    dfl, lv = worlds.add_context_levels(df, KNOTS, DOY_K, DOY_F, hour_season=True, solar=True)
    sp = splits.kfold(df.year.to_numpy())
    D = worlds.split_levels(df, sp, KNOTS, DOY_K, DOY_F, hour_season=True, solar=True)[0][level]
    wk = df.year.to_numpy() * 14 + (df.doy.to_numpy() - DOY0) // 7
    p = inference.perm_test(D, df[W['vis']].to_numpy(), df.d_ref_k.to_numpy(), sp, wk, df.hour_local.to_numpy(), keep=scoring.support_mask(df[['hour_local', 'temp_cond_c']].to_numpy(), sp),
                            units=df.day.to_numpy(), strata=df.year.to_numpy(), Cnull=dfl[lv[level]].to_numpy(), clip=.005, robust=True, shift_min=7, max_shifts=N_NULL)[1]
    return dict(est=out['est'], p=p)

FORCED = {k: DEV[k] for k in ['D-N2', 'D-N3', 'D-N4', 'D-P3']}
forced = pd.DataFrame([dict(world=k, context=lv, rej=(a.p <= .05).mean(), est=a.est.mean()) for k, W in FORCED.items() for lv in ['L1', 'L3', 'L4', 'S', 'S4']
                       for a in [pd.DataFrame(par()(delayed(forced_rep)(W, lv, 650000 + j) for j in range(40)))]]).set_index(['world', 'context'])
forced.unstack('context')

rej                                    est                        \
context     L1     L3     L4      S     S4         L1         L3         L4   
world                                                                         
D-N2     0.075  0.050  0.025  0.025  0.050 -9.484e-04 -1.090e-03 -3.996e-04   
D-N3     0.975  0.700  0.075  0.175  0.050  1.077e-02  4.405e-03  1.073e-04   
D-N4     0.025  0.075  0.050  0.075  0.025 -1.966e-04 -2.756e-04 -2.526e-04   
D-P3     1.000  1.000  1.000  1.000  0.975  2.729e-02  2.242e-02  2.089e-02   

                               
context          S         S4  
world                          
D-N2    -3.505e-04 -7.730e-04  
D-N3     4.592e-03  1.144e-04  
D-N4    -2.508e-04 -1.260e-04  
D-P3     1.868e-02  1.514e-02

### Auditor thresholds (development diagnostic)

Both auditors flag when their one-sided 95 % week-cluster lower bound exceeds a materiality threshold τ (nats per row).
- With τ = 0 the repair diagnostic raised false alarms in adequately modelled worlds (the table above).
- The flag rates below are read from the stored lower bounds of the same development replicates, under the chosen context and under the deliberately misspecified L1.
- A threshold is chosen here, on development worlds only, and frozen for notebook 16.

In [5]:
TAUS = [0.0, 0.001, 0.002, 0.003, 0.005]
pd.concat({f'tau={t}': pd.DataFrame({k: {'audit (chosen)': (a.audit_lb > t).mean(), 'audit (L1)': (a.audit_lb_L1 > t).mean(),
                                         'repair (chosen)': (a.repair_lb > t).mean(), 'repair (L1)': (a.repair_lb_L1 > t).mean()} for k, a in dev_raw.items()}).T for t in TAUS}, axis=1).round(3)

tau=0.0                                             tau=0.001  \
     audit (chosen) audit (L1) repair (chosen) repair (L1) audit (chosen)   
D-N1          0.050      0.000           0.075       0.050          0.050   
D-N2          0.000      0.025           0.100       0.100          0.000   
D-N3          0.075      0.550           0.275       1.000          0.050   
D-N4          0.050      0.000           0.025       0.025          0.050   
D-N5          0.100      0.150           0.125       0.100          0.100   
D-N6          0.050      0.075           0.075       0.125          0.050   
D-N7          0.025      0.025           0.050       0.025          0.025   
D-N8          0.050      0.025           0.150       0.050          0.050   
D-P1          0.000      0.000           0.100       0.000          0.000   
D-P2          0.050      0.075           0.000       0.000          0.050   
D-P3          0.025      0.025           0.025       0.050          0.025   
D-P4          0.075      0.175           0.025       0.000          0.050   

                                                 tau=0.002             \
     audit (L1) repair (chosen) repair (L1) audit (chosen) audit (L1)   
D-N1      0.000           0.000       0.025          0.025      0.000   
D-N2      0.000           0.000       0.000          0.000      0.000   
D-N3      0.525           0.000       1.000          0.000      0.475   
D-N4      0.000           0.000       0.000          0.025      0.000   
D-N5      0.100           0.075       0.075          0.075      0.100   
D-N6      0.050           0.075       0.075          0.050      0.050   
D-N7      0.025           0.025       0.000          0.025      0.025   
D-N8      0.025           0.025       0.025          0.025      0.000   
D-P1      0.000           0.050       0.000          0.000      0.000   
D-P2      0.025           0.000       0.000          0.050      0.025   
D-P3      0.025           0.000       0.025          0.025      0.000   
D-P4      0.150           0.000       0.000          0.025      0.125   

                                      tau=0.003                             \
     repair (chosen) repair (L1) audit (chosen) audit (L1) repair (chosen)   
D-N1           0.000       0.000          0.000      0.000            0.00   
D-N2           0.000       0.000          0.000      0.000            0.00   
D-N3           0.000       1.000          0.000      0.375            0.00   
D-N4           0.000       0.000          0.025      0.000            0.00   
D-N5           0.075       0.000          0.025      0.100            0.05   
D-N6           0.075       0.075          0.050      0.050            0.05   
D-N7           0.025       0.000          0.000      0.025            0.00   
D-N8           0.025       0.000          0.025      0.000            0.00   
D-P1           0.000       0.000          0.000      0.000            0.00   
D-P2           0.000       0.000          0.000      0.025            0.00   
D-P3           0.000       0.000          0.000      0.000            0.00   
D-P4           0.000       0.000          0.025      0.100            0.00   

                      tau=0.005                                         
     repair (L1) audit (chosen) audit (L1) repair (chosen) repair (L1)  
D-N1        0.00          0.000      0.000           0.000       0.000  
D-N2        0.00          0.000      0.000           0.000       0.000  
D-N3        1.00          0.000      0.275           0.000       0.925  
D-N4        0.00          0.025      0.000           0.000       0.000  
D-N5        0.00          0.025      0.050           0.050       0.000  
D-N6        0.05          0.025      0.050           0.025       0.050  
D-N7        0.00          0.000      0.025           0.000       0.000  
D-N8        0.00          0.025      0.000           0.000       0.000  
D-P1        0.00          0.000      0.000           0.000       0.000  
D-P2        0.

## 2 · Development reading

These are development results (40 replicates per world, Monte Carlo SE of a 0.05 rate 0.034). They choose the protocol; they do not validate it. The tables reproduce the previous development run exactly after the machinery moved into `visual_hygrometry.suite`.

**Context.**
- The forced diagnostic shows why notebook 11 failed: in D-N3 the clock-time contexts reject at 0.975 (L1) and 0.70 (L3), and the solar terms alone (S) still reject at 0.175. Only the hour × season products (L4, S4) control it (0.075, 0.05).
- D-N2, D-N4 and the sun-clock positive D-P3 behave under every candidate, so the solar terms are not needed for them, but they cost little power (D-P3 0.975–1.0).
- With selection by grouped inner CV on training rows only, D-N3 rejects at 0.025 under both generators. Selection is therefore doing the work that review 5 asked of it; the solar basis on its own is not sufficient, which is why S4 stays in the family.

**Null generator: CRT.**
- The shift test loses Type I control under persistence: 0.175 at persistence 0.95 (D-N5), 0.10 at 0.98 (D-N6) and 0.125 with season-specific vegetation and a camera change (D-N8).
- The CRT stays at or below 0.075 in every null world and has equal or higher power in every positive world (D-P2 0.80 against 0.325; D-P4 0.775 against 0.675).
- The CRT is chosen for notebook 16. Its validity rests on the sampler's model of V given the raw context and season (day-level and within-day AR(1) per season). It is not exact, and notebook 16 tests it on persistence, regimes and vegetation structures it was not tuned on.

**Estimates.**
- In positive worlds the estimate tracks G_design (|est − G| ≤ 0.003) and sits below E2 by 0.007–0.011 nats: G for three training seasons is smaller than E2 here, as the statistical note expects for a finite design.
- In persistent nulls the estimate is negative (−0.014, −0.037), because a model fitted on persistent noise does not transfer to a new season. A negative contrast cannot produce a false claim, so gate A3 is one-sided in nulls.

**Intervals: not yet adequate.** The season-cluster interval covers G_design in 0.90–1.0 of nulls without persistence, but only 0.775 and 0.825 under persistence and 0.70–0.90 in positive worlds. Three degrees of freedom do not make it valid. Gate A4 is kept at 0.92 as review 5 requires; on this evidence it is expected to fail, and season-specific G is reported descriptively.

**Stability.** The 90th percentile of the change after deleting 1 % of rows is ≤ 0.024 nats, the largest under persistence.

**Auditors and thresholds (frozen for notebook 16).**
- Both auditors replace first versions that failed in an earlier development run: boosted trees on the raw context memorised weather through day of year (the challenger's inner MSE rose from 4.8 to 6.2), and the boosted repair projection did the same. The smooth ridge versions above are what was evaluated.
- *Challenger, τ_audit = 0.* False alarms are ≤ 0.10 on the chosen context at τ = 0, and any positive τ only reduces detection (D-N3 under L1: 0.55 at τ = 0, 0.475 at 0.002, 0.275 at 0.005). Its power against the catastrophic misspecification is too low; gate A6 keeps the 0.80 detection requirement and is expected to fail on this evidence.
- *Repair diagnostic, τ_repair = 0.002 nats per row.* At τ = 0 it raised false alarms on adequate contexts (0.275 in D-N3, 0.15 in D-N8). At τ = 0.002 false alarms are ≤ 0.075 in every world and detection under L1 in D-N3 stays at 1.0. It remains a secondary diagnostic, because a projection E[V | C] can absorb slow visual information.
- Detection was measured on one misspecified world only (D-N3; L1 is adequate in D-N2 and D-N4). Notebook 16 adds a second misspecified null with a day-length-dependent diurnal amplitude (V-N4).